In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import copy

In [2]:
FEATURE_COLS = ['logon_count', 'logoff_count', 'unique_pcs',
                'first_logon_hour', 'last_logoff_hour', 'active_hours']
N_ESTIMATORS_PER_NODE = 50
MIN_DAYS_PER_NODE = 10
REFERENCE_DAYS_PER_NODE = 5

In [3]:
raw_df = pd.read_csv('logon.csv')
raw_df['date'] = pd.to_datetime(raw_df['date'])
raw_df['day'] = raw_df['date'].dt.date
raw_df['hour'] = raw_df['date'].dt.hour

features_df = raw_df.groupby(['user', 'day']).agg(
    logon_count=('activity', lambda x: (x == 'Logon').sum()),
    logoff_count=('activity', lambda x: (x == 'Logoff').sum()),
    unique_pcs=('pc', 'nunique'),
    first_logon_hour=('hour', 'min'),
    last_logoff_hour=('hour', 'max'),
    active_hours=('hour', 'nunique')
).reset_index()

print(features_df.shape)
features_df.head()

(1394010, 8)


,user,day,logon_count,logoff_count,unique_pcs,first_logon_hour,last_logoff_hour,active_hours
0,AAB0162,2010-01-04,1,1,1,7,18,2
1,AAB0162,2010-01-05,1,1,1,7,18,2
2,AAB0162,2010-01-06,1,1,1,7,18,2
3,AAB0162,2010-01-07,1,1,1,7,18,2
4,AAB0162,2010-01-08,1,1,1,7,18,2


In [4]:
sample_users = features_df['user'].unique()[:50]
user_data = {
    user: group[FEATURE_COLS].values
    for user, group in features_df[features_df['user'].isin(sample_users)].groupby('user')
}
print(f"Nodes: {len(user_data)}")

Nodes: 50


In [5]:
rng = np.random.default_rng(42)
user_data['INSIDER_SYNTHETIC'] = np.column_stack([
    rng.integers(3, 6, 30),    # logon_count
    rng.integers(1, 3, 30),    # logoff_count
    rng.integers(2, 4, 30),    # unique_pcs
    rng.integers(0, 5, 30),    # first_logon_hour
    rng.integers(21, 24, 30),  # last_logoff_hour
    rng.integers(5, 8, 30),    # active_hours
]).astype(float)

print("Insider injected. Total nodes:", len(user_data))

Insider injected. Total nodes: 51


In [6]:
all_data = np.vstack(list(user_data.values()))
global_scaler = StandardScaler()
global_scaler.fit(all_data)
print("Scaler fitted on shape:", all_data.shape)

Scaler fitted on shape: (17462, 6)


In [7]:
ref_rows = []
for user, logs in user_data.items():
    if user == 'INSIDER_SYNTHETIC':
        continue
    ref_rows.append(logs[:REFERENCE_DAYS_PER_NODE])

reference_scaled = global_scaler.transform(np.vstack(ref_rows))
print("Reference dataset shape:", reference_scaled.shape)

Reference dataset shape: (250, 6)


In [8]:
local_models = []
skipped = 0

for user, logs in user_data.items():
    if len(logs) < MIN_DAYS_PER_NODE:
        skipped += 1
        continue
    scaled = global_scaler.transform(logs)
    combined = np.vstack([scaled, reference_scaled])
    model = IsolationForest(n_estimators=N_ESTIMATORS_PER_NODE,
                            contamination='auto', random_state=42)
    model.fit(combined)
    local_models.append(model)

print(f"Trained: {len(local_models)}, Skipped: {skipped}")

Trained: 51, Skipped: 0


In [9]:
global_model = copy.deepcopy(local_models[0])
all_trees, all_samples = [], []

for m in local_models:
    all_trees += m.estimators_
    all_samples += m.__dict__.get('estimators_samples_', [])

global_model.estimators_ = all_trees
global_model.n_estimators = len(all_trees)
global_model.max_samples_ = local_models[0].max_samples_
global_model.__dict__['estimators_samples_'] = all_samples

print(f"Global model trees: {global_model.n_estimators}")

Global model trees: 2550


In [10]:
results = []
for user, logs in user_data.items():
    if len(logs) < MIN_DAYS_PER_NODE:
        continue
    scaled = global_scaler.transform(logs)
    scores = global_model.decision_function(scaled)
    labels = global_model.predict(scaled)
    results.append({
        'user': user,
        'total_days': len(logs),
        'anomaly_days': int((labels == -1).sum()),
        'anomaly_rate': round((labels == -1).mean(), 3),
        'avg_anomaly_score': round(float(scores.mean()), 4),
        'min_anomaly_score': round(float(scores.min()), 4)
    })

rankings = pd.DataFrame(results).sort_values('avg_anomaly_score')
rankings.head(10)

,user,total_days,anomaly_days,anomaly_rate,avg_anomaly_score,min_anomaly_score
50,INSIDER_SYNTHETIC,30,30,1.0,-0.4960,-0.4964
38,ABM3203,438,438,1.0,-0.4926,-0.4957
46,ABR0397,356,356,1.0,-0.4912,-0.4952
25,ABB0300,384,384,1.0,-0.4909,-0.4947
40,ABM3687,216,216,1.0,-0.4909,-0.4950
1,AAB0398,356,356,1.0,-0.4906,-0.4906
36,ABK3081,428,428,1.0,-0.4906,-0.4959
16,AAP1942,356,356,1.0,-0.4905,-0.4923
41,ABM3772,363,363,1.0,-0.4900,-0.4961
43,ABO1173,363,363,1.0,-0.4893,-0.4961


In [11]:
def is_insider_threat(log_entry: dict) -> dict:
    try:
        features = np.array([[log_entry[f] for f in FEATURE_COLS]])
    except KeyError as e:
        return {"error": f"Missing feature: {e}"}

    scaled = global_scaler.transform(features)
    score = global_model.decision_function(scaled)[0]
    label = global_model.predict(scaled)[0]

    return {
        "is_anomaly": bool(label == -1),
        "anomaly_score": round(float(score), 4),
        "node_id": log_entry.get("node_id", "unknown")
    }

# Test it
test_log = {
    "node_id": "node_test",
    "logon_count": 5,
    "logoff_count": 1,
    "unique_pcs": 3,
    "first_logon_hour": 2,
    "last_logoff_hour": 23,
    "active_hours": 7
}

is_insider_threat(test_log)

{'is_anomaly': True, 'anomaly_score': -0.4957, 'node_id': 'node_test'}

In [12]:
import joblib
joblib.dump(global_model, 'gap3_model.pkl')
joblib.dump(global_scaler, 'gap3_scaler.pkl')

['gap3_scaler.pkl']